# Silver Layer: Wind Turbine Data Cleaning

Reads from `bronze.wind_turbine_raw`, applies null handling and outlier removal, and writes cleaned rows to `silver.wind_turbine_clean`.

## Cleaning steps

### 1. Null handling
| Column | Strategy | Rationale |
|---|---|---|
| `timestamp` | **Drop row** | A reading with no time context is unplaceable in any time-series analysis |
| `turbine_id` | **Drop row** | Cannot attribute a reading to any turbine |
| `power_output_mw` | **Drop row** | Core metric — imputing power from wind speed would introduce modelling assumptions outside the pipeline's scope |
| `wind_speed_ms` | **Impute with per-turbine daily median** | Commonly available from nearby sensors; median is robust to remaining outliers |
| `wind_direction_deg` | **Impute with per-turbine daily median** | Same rationale as wind speed |

### 2. Outlier detection and removal
Outliers are defined per-turbine using a rolling **±2 standard deviation** window over each numeric column.  
Rows where any numeric column falls outside `[mean − 2σ, mean + 2σ]` are flagged with `_is_outlier = true` and excluded from the clean output.

Stats are computed **per turbine per day** to account for naturally different operating baselines across turbines and diurnal wind patterns.

### 3. Physical range guards
Hard physical limits are applied as a first pass before statistical outlier detection:
- `wind_speed_ms` ∈ [0, 50] — typical cut-out speed is ~25 m/s; 50 is a generous upper bound
- `wind_direction_deg` ∈ [0, 360]
- `power_output_mw` ≥ 0 — negative power is physically impossible

In [ ]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("catalog", "wind_farm_dev")
dbutils.widgets.text("schema",  "wind_farm")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

bronze_table = f"`{catalog}`.`{schema}`.wind_turbine_raw"
silver_table = f"`{catalog}`.`{schema}`.wind_turbine_clean"

# Number of standard deviations that defines an outlier
OUTLIER_STD_THRESHOLD = 2.0

# Hard physical bounds — rows outside these are impossible sensor values
PHYSICAL_BOUNDS = {
    "wind_speed":      (0.0,  50.0),
    "wind_direction": (0.0, 360.0),
    "power_output":    (0.0,  None),   # no hard upper bound
}

print(f"Source : {bronze_table}")
print(f"Target : {silver_table}")

In [ ]:
from pyspark.sql import functions as F, Window

bronze_df = spark.table(bronze_table)
print(f"Bronze row count: {bronze_df.count():,}")

## Step 1 — Drop rows with null anchor columns

In [ ]:
ANCHOR_COLS = ["timestamp", "turbine_id", "power_output"]

display(bronze_df.limit(20))

after_anchor_drop = bronze_df.dropna(subset=ANCHOR_COLS)

display(after_anchor_drop.limit(20))

dropped_anchor = bronze_df.count() - after_anchor_drop.count()
print(f"Rows dropped (null anchor column): {dropped_anchor:,}")

## Step 2 — Physical range filter

In [ ]:
range_filter = after_anchor_drop

for col_name, (lower, upper) in PHYSICAL_BOUNDS.items():
    # Only filter non-null values, nulls not yet imputed are preserved here
    if lower is not None:
        range_filter = range_filter.filter(
            F.col(col_name).isNull() | (F.col(col_name) >= lower)
        )
    if upper is not None:
        range_filter = range_filter.filter(
            F.col(col_name).isNull() | (F.col(col_name) <= upper)
        )

dropped_range = after_anchor_drop.count() - range_filter.count()
print(f"Rows dropped (outside physical bounds): {dropped_range:,}")

## Step 3 — Impute remaining nulls with per-turbine daily median

In [ ]:
IMPUTE_COLS = ["wind_speed", "wind_direction"]

# Partition key: turbine + calendar day
daily_window = Window.partitionBy("turbine_id", F.to_date("timestamp"))

imputed = range_filter
for col_name in IMPUTE_COLS:
    median_col = f"_median_{col_name}"
    imputed = (
        imputed
            .withColumn(median_col, F.percentile_approx(col_name, 0.5).over(daily_window))
            .withColumn(col_name, F.coalesce(F.col(col_name), F.col(median_col)))
            .drop(median_col)
    )

# Any rows where the whole day's data for that turbine was null remain null;
# drop them as they cannot be meaningfully imputed.
after_impute = imputed.dropna(subset=IMPUTE_COLS)

dropped_impute = range_filter.count() - after_impute.count()
print(f"Rows dropped (un-imputable nulls after median fill): {dropped_impute:,}")

## Step 4 — Statistical outlier detection (±2σ per turbine per day)

In [ ]:
NUMERIC_COLS = ["wind_speed", "wind_direction", "power_output"]

stats_window = Window.partitionBy("turbine_id", F.to_date("timestamp"))

with_stats = after_impute
outlier_flags = []

for col_name in NUMERIC_COLS:
    mean_col  = f"_mean_{col_name}"
    std_col   = f"_std_{col_name}"
    flag_col  = f"_outlier_{col_name}"

    with_stats = (
        with_stats
            .withColumn(mean_col, F.avg(col_name).over(stats_window))
            .withColumn(std_col,  F.stddev(col_name).over(stats_window))
    )

    # A row is an outlier if it falls outside mean +/- (threshold * std).
    # When std is null (only one reading that day) we cannot compute bounds
    # so we conservatively keep the row.
    with_stats = with_stats.withColumn(
        flag_col,
        F.when(
            F.col(std_col).isNull(),
            F.lit(False)
        ).otherwise(
            (F.col(col_name) < (F.col(mean_col) - OUTLIER_STD_THRESHOLD * F.col(std_col))) |
            (F.col(col_name) > (F.col(mean_col) + OUTLIER_STD_THRESHOLD * F.col(std_col)))
        )
    )
    outlier_flags.append(flag_col)

display(with_stats)

# Combine per-column flags: a row is an outlier if any column is flagged
combined_flag = F.col(outlier_flags[0])
for flag in outlier_flags[1:]:
    combined_flag = combined_flag | F.col(flag)

with_outlier_col = with_stats.withColumn("_is_outlier", combined_flag)

display(with_outlier_col)

outlier_count = with_outlier_col.filter(F.col("_is_outlier")).count()
print(f"Outlier rows flagged: {outlier_count:,}")

## Step 5 — Build silver output

Drop all intermediate stat/flag columns and exclude outlier rows from the clean table.

In [ ]:
# Columns to drop before writing (stats were only needed for flagging)
temp_cols = (
    [f"_mean_{c}" for c in NUMERIC_COLS]
    + [f"_std_{c}"  for c in NUMERIC_COLS]
    + outlier_flags
)

silver_df = (
    with_outlier_col
        .filter(~F.col("_is_outlier"))
        .drop(*temp_cols, "_is_outlier")
        .withColumn("_cleaned_at", F.current_timestamp())
)

print(f"Silver row count: {silver_df.count():,}")

## Step 6 — Write silver Delta table

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

# (
#     silver_df.write
#         .format("delta")
#         .mode("overwrite")              # Full reprocess on each run; safe because
#         .option("overwriteSchema", "true")  # bronze is the system of record.
#         .saveAsTable(silver_table)
# )

# print(f"Silver write complete → {silver_table}")

## Step 7 — Data quality summary

In [ ]:
bronze_count = bronze_df.count()
silver_count = spark.table(silver_table).count()
retention_pct = 100 * silver_count / bronze_count if bronze_count else 0

print("=" * 50)
print(f"Bronze rows              : {bronze_count:>10,}")
print(f"  Dropped — null anchor  : {dropped_anchor:>10,}")
print(f"  Dropped — range bounds : {dropped_range:>10,}")
print(f"  Dropped — un-imputable : {dropped_impute:>10,}")
print(f"  Dropped — outliers     : {outlier_count:>10,}")
print(f"Silver rows              : {silver_count:>10,}  ({retention_pct:.1f}% retained)")
print("=" * 50)

print("\nNull counts remaining in silver:")
spark.table(silver_table).select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in NUMERIC_COLS + ["timestamp", "turbine_id"]]
).show()